# Instalment Payment Feature Engineering

This notebook builds applicant-level features from the cleaned instalment-payment data. It first consolidates split payment entries into one row per scheduled instalment, then creates payment-timing and payment-amount features, and aggregates everything to one row per applicant.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
input_path = project_root / "data" / "interim" / "installments_payments_clean.pkl"
application_path = project_root / "data" / "interim" / "application_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "features" / "installment_payment_features.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [input_path, application_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Clean input:", input_path)
print("Feature output:", output_path)

Clean input: /Users/taranveersingh/A-MRP/data/interim/installments_payments_clean.pkl
Feature output: /Users/taranveersingh/A-MRP/data/features/installment_payment_features.pkl


## Load cleaned payments and split information


In [7]:
payments = pd.read_pickle(input_path)
application_target = pd.read_pickle(application_path)[["SK_ID_CURR", "TARGET"]]
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)
print("Clean payment-entry shape:", payments.shape)
print("Applicants represented:", payments["SK_ID_CURR"].nunique())
print("Previous loans represented:", payments["SK_ID_PREV"].nunique())

Clean payment-entry shape: (11591592, 14)
Applicants represented: 291643
Previous loans represented: 853344


## Consolidate split payments to one installment record


In [9]:
installment_keys = [
    "SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "NUM_INSTALMENT_VERSION"
]
grouped_payments = payments.groupby(installment_keys, sort=False)
installment_level = grouped_payments.agg(
    DAYS_INSTALMENT=("DAYS_INSTALMENT", "first"),
    DAYS_ENTRY_PAYMENT=("DAYS_ENTRY_PAYMENT", "max"),
    AMT_INSTALMENT=("AMT_INSTALMENT", "max"),
    PAYMENT_ENTRY_COUNT=("SK_ID_PREV", "size"),
    PAYMENT_RECORD_MISSING_RATE_MEAN=("INST_RECORD_MISSING_RATE", "mean"),
).reset_index()
payment_totals = grouped_payments["AMT_PAYMENT"].sum(min_count=1).rename("AMT_PAYMENT_TOTAL").reset_index()
installment_level = installment_level.merge(payment_totals, on=installment_keys, how="left", validate="one_to_one")
raw_payment_rows = len(payments)
split_payment_installments = int(installment_level["PAYMENT_ENTRY_COUNT"].gt(1).sum())
del grouped_payments, payment_totals, payments
print("Raw payment-entry rows:", raw_payment_rows)
print("Unique consolidated installments:", len(installment_level))
print("Installments with multiple payment entries:", split_payment_installments)

Raw payment-entry rows: 11591592
Unique consolidated installments: 11026627
Installments with multiple payment entries: 553925


About 554,000 scheduled instalments had more than one payment entry, meaning the payment was made in separate parts. These are combined into a single row per instalment so they do not get counted more than once.


## Create accurate installment-level payment features


In [12]:
def safe_ratio(numerator, denominator):
    return (numerator / denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

installment_level["INST_DAYS_LATE"] = (
    installment_level["DAYS_ENTRY_PAYMENT"] - installment_level["DAYS_INSTALMENT"]
).clip(lower=0)
installment_level["INST_DAYS_EARLY"] = (
    installment_level["DAYS_INSTALMENT"] - installment_level["DAYS_ENTRY_PAYMENT"]
).clip(lower=0)
installment_level["INST_PAYMENT_DIFFERENCE"] = installment_level["AMT_INSTALMENT"] - installment_level["AMT_PAYMENT_TOTAL"]
installment_level["INST_PAYMENT_RATIO"] = safe_ratio(installment_level["AMT_PAYMENT_TOTAL"], installment_level["AMT_INSTALMENT"])
payment_information_missing = installment_level[["DAYS_ENTRY_PAYMENT", "AMT_PAYMENT_TOTAL"]].isna().any(axis=1)
installment_level["INST_PAYMENT_INFORMATION_MISSING"] = payment_information_missing.astype("int8")

# Missing payment information is not labelled as on-time or fully paid.
installment_level["INST_IS_LATE"] = installment_level["INST_DAYS_LATE"].gt(0).where(~payment_information_missing, np.nan).astype("float32")
installment_level["INST_IS_SEVERELY_LATE"] = installment_level["INST_DAYS_LATE"].gt(30).where(~payment_information_missing, np.nan).astype("float32")
installment_level["INST_IS_UNDERPAID"] = installment_level["INST_PAYMENT_DIFFERENCE"].gt(0.01).where(~payment_information_missing, np.nan).astype("float32")
installment_level["INST_IS_OVERPAID"] = installment_level["INST_PAYMENT_DIFFERENCE"].lt(-0.01).where(~payment_information_missing, np.nan).astype("float32")
installment_level["INST_IS_ZERO_PAYMENT"] = installment_level["AMT_PAYMENT_TOTAL"].eq(0).astype("int8")
installment_level["INST_HAS_SPLIT_PAYMENT"] = installment_level["PAYMENT_ENTRY_COUNT"].gt(1).astype("int8")
installment_level["INST_RECENT_12M"] = installment_level["DAYS_INSTALMENT"].ge(-365).astype("int8")
print("Installment-level features created: 12")
print("Late installments after consolidation:", int(installment_level["INST_IS_LATE"].sum()))
print("Underpaid installments after consolidation:", int(installment_level["INST_IS_UNDERPAID"].sum()))

Installment-level features created: 12
Late installments after consolidation: 955113
Underpaid installments after consolidation: 2621


These measure how late or early each instalment was paid, and whether it was underpaid, based on the consolidated payment amounts.


## Aggregate full payment history to applicant level


In [15]:
installment_features = installment_level.groupby("SK_ID_CURR").agg(
    INST_INSTALLMENT_COUNT=("NUM_INSTALMENT_NUMBER", "count"),
    INST_PREVIOUS_LOAN_COUNT=("SK_ID_PREV", "nunique"),
    INST_PAYMENT_ENTRY_COUNT=("PAYMENT_ENTRY_COUNT", "sum"),
    INST_SPLIT_PAYMENT_COUNT=("INST_HAS_SPLIT_PAYMENT", "sum"),
    INST_SPLIT_PAYMENT_RATE=("INST_HAS_SPLIT_PAYMENT", "mean"),
    INST_LATE_COUNT=("INST_IS_LATE", "sum"),
    INST_LATE_RATE=("INST_IS_LATE", "mean"),
    INST_SEVERELY_LATE_COUNT=("INST_IS_SEVERELY_LATE", "sum"),
    INST_SEVERELY_LATE_RATE=("INST_IS_SEVERELY_LATE", "mean"),
    INST_DAYS_LATE_MEAN=("INST_DAYS_LATE", "mean"),
    INST_DAYS_LATE_MAX=("INST_DAYS_LATE", "max"),
    INST_DAYS_LATE_SUM=("INST_DAYS_LATE", "sum"),
    INST_DAYS_EARLY_MEAN=("INST_DAYS_EARLY", "mean"),
    INST_DAYS_EARLY_MAX=("INST_DAYS_EARLY", "max"),
    INST_UNDERPAID_COUNT=("INST_IS_UNDERPAID", "sum"),
    INST_UNDERPAID_RATE=("INST_IS_UNDERPAID", "mean"),
    INST_OVERPAID_COUNT=("INST_IS_OVERPAID", "sum"),
    INST_OVERPAID_RATE=("INST_IS_OVERPAID", "mean"),
    INST_ZERO_PAYMENT_COUNT=("INST_IS_ZERO_PAYMENT", "sum"),
    INST_MISSING_PAYMENT_COUNT=("INST_PAYMENT_INFORMATION_MISSING", "sum"),
    INST_MISSING_PAYMENT_RATE=("INST_PAYMENT_INFORMATION_MISSING", "mean"),
    INST_SCHEDULED_AMOUNT_TOTAL=("AMT_INSTALMENT", "sum"),
    INST_PAID_AMOUNT_TOTAL=("AMT_PAYMENT_TOTAL", "sum"),
    INST_PAYMENT_DIFFERENCE_TOTAL=("INST_PAYMENT_DIFFERENCE", "sum"),
    INST_PAYMENT_DIFFERENCE_MEAN=("INST_PAYMENT_DIFFERENCE", "mean"),
    INST_PAYMENT_DIFFERENCE_MAX=("INST_PAYMENT_DIFFERENCE", "max"),
    INST_PAYMENT_RATIO_MEAN=("INST_PAYMENT_RATIO", "mean"),
    INST_PAYMENT_RATIO_MIN=("INST_PAYMENT_RATIO", "min"),
    INST_RECORD_MISSING_RATE_MEAN=("PAYMENT_RECORD_MISSING_RATE_MEAN", "mean"),
).reset_index()
installment_features["INST_TOTAL_PAYMENT_RATIO"] = safe_ratio(
    installment_features["INST_PAID_AMOUNT_TOTAL"], installment_features["INST_SCHEDULED_AMOUNT_TOTAL"]
)
print("General applicant feature shape:", installment_features.shape)

General applicant feature shape: (291643, 31)


Each applicant's instalment history is summarized here: counts and rates of late payments, underpayments, and split payments, along with average and maximum days late.


## Add recent 12-month payment behaviour


In [18]:
recent_installments = installment_level.loc[installment_level["INST_RECENT_12M"].eq(1)]
recent_features = recent_installments.groupby("SK_ID_CURR").agg(
    INST_RECENT_12M_COUNT=("NUM_INSTALMENT_NUMBER", "count"),
    INST_RECENT_12M_LATE_COUNT=("INST_IS_LATE", "sum"),
    INST_RECENT_12M_LATE_RATE=("INST_IS_LATE", "mean"),
    INST_RECENT_12M_DAYS_LATE_MAX=("INST_DAYS_LATE", "max"),
    INST_RECENT_12M_UNDERPAID_COUNT=("INST_IS_UNDERPAID", "sum"),
    INST_RECENT_12M_UNDERPAID_RATE=("INST_IS_UNDERPAID", "mean"),
    INST_RECENT_12M_PAYMENT_RATIO_MEAN=("INST_PAYMENT_RATIO", "mean"),
).reset_index()
installment_features = installment_features.merge(recent_features, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Applicants with recent installment history:", len(recent_features))
print("Applicant feature shape:", installment_features.shape)

Applicants with recent installment history: 217508
Applicant feature shape: (291643, 38)


Same kind of summary as before, but limited to the most recent 12 months of instalments, to capture more current repayment behaviour separately from the full history.


## Apply training-only missingness and constant-feature rules


In [21]:
MISSING_THRESHOLD = 0.50
training_base = pd.DataFrame({"SK_ID_CURR": training_ids}).merge(
    application_target, on="SK_ID_CURR", how="left", validate="one_to_one"
).merge(installment_features, on="SK_ID_CURR", how="left", validate="one_to_one")
decision_rows = []
for feature in [c for c in installment_features.columns if c != "SK_ID_CURR"]:
    series = training_base[feature]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    correlation = series.corr(training_base["TARGET"]) if unique_non_missing > 1 else np.nan
    decision = "Keep"
    reason = "Retain for global cross-validated feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-applicant missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set where values are available"
    decision_rows.append({
        "feature": feature, "missing_count": int(series.isna().sum()),
        "missing_rate": missing_rate, "unique_non_missing": int(unique_non_missing),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "decision": decision, "reason": reason,
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "absolute_correlation"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
installment_features = installment_features.drop(columns=removed_features)
print("Features removed:", removed_features)
print("Features retained:", installment_features.shape[1] - 1)
feature_decisions.round(5)

Features removed: []
Features retained: 37


,feature,missing_count,missing_rate,unique_non_missing,pearson_target_correlation,absolute_correlation,decision,reason
0,INST_RECENT_12M_LATE_RATE,72189,0.29344,549,0.08079,0.08079,Keep,Retain for global cross-validated feature sele...
1,INST_RECENT_12M_LATE_COUNT,72166,0.29335,41,0.07739,0.07739,Keep,Retain for global cross-validated feature sele...
2,INST_RECENT_12M_DAYS_LATE_MAX,72189,0.29344,181,0.07246,0.07246,Keep,Retain for global cross-validated feature sele...
3,INST_LATE_RATE,12725,0.05173,4362,0.07160,0.07160,Keep,Retain for global cross-validated feature sele...
4,INST_SPLIT_PAYMENT_RATE,12718,0.05170,3267,0.06060,0.06060,Keep,Retain for global cross-validated feature sele...
5,INST_RECENT_12M_UNDERPAID_COUNT,72166,0.29335,9,0.03730,0.03730,Keep,Retain for global cross-validated feature sele...
6,INST_PREVIOUS_LOAN_COUNT,12718,0.05170,25,-0.03334,0.03334,Keep,Retain for global cross-validated feature sele...
7,INST_SEVERELY_LATE_RATE,12725,0.05173,857,0.03233,0.03233,Keep,Retain for global cross-validated feature sele...
8,INST_LATE_COUNT,12718,0.05170,91,0.03023,0.03023,Keep,Retain for global cross-validated feature sele...
9,INST_DAYS_EARLY_MEAN,12725,0.05173,48687,-0.02977,0.02977,Keep,Retain for global cross-validated feature sele...


No instalment features were removed here, all 37 stayed within the missing-value and constant-value limits.


## Validate the applicant-level feature table


In [24]:
numeric_columns = installment_features.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(installment_features[c].dropna()).sum()) for c in numeric_columns)
retained_decisions = feature_decisions.loc[feature_decisions["decision"] == "Keep"]
validation_checks = pd.DataFrame([
    {"check": "One row per applicant", "passed": installment_features["SK_ID_CURR"].is_unique},
    {"check": "Only project applicants included", "passed": set(installment_features["SK_ID_CURR"]).issubset(training_id_set.union(test_id_set))},
    {"check": "No TARGET in output", "passed": "TARGET" not in installment_features.columns},
    {"check": "No previous-loan ID in output", "passed": "SK_ID_PREV" not in installment_features.columns},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "No retained feature reaches 50 percent training missingness", "passed": not retained_decisions["missing_rate"].ge(MISSING_THRESHOLD).any()},
    {"check": "Final test excluded from feature decisions", "passed": not training_base["SK_ID_CURR"].isin(test_id_set).any()},
    {"check": "Consolidated installments do not exceed payment rows", "passed": len(installment_level) <= raw_payment_rows},
    {"check": "Installment counts are positive", "passed": installment_features["INST_INSTALLMENT_COUNT"].gt(0).all()},
])
assert validation_checks["passed"].all(), "At least one installment feature-engineering check failed."
validation_checks

,check,passed
0,One row per applicant,True
1,Only project applicants included,True
2,No TARGET in output,True
3,No previous-loan ID in output,True
4,No infinite numerical values,True
5,No retained feature reaches 50 percent trainin...,True
6,Final test excluded from feature decisions,True
7,Consolidated installments do not exceed paymen...,True
8,Installment counts are positive,True


All checks passed.


## Save features and audit reports


In [27]:
installment_features.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "installment_payment_engineered_feature_decisions.csv", index=False)
validation_checks.to_csv(audit_folder / "installment_payment_feature_engineering_validation.csv", index=False)
print("Installment-payment feature table saved:", output_path)
print("Output rows:", len(installment_features))
print("Output columns:", installment_features.shape[1])
print("Applicants represented:", installment_features["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(installment_features.select_dtypes(include="number").isna().sum().sum()))

Installment-payment feature table saved: /Users/taranveersingh/A-MRP/data/features/installment_payment_features.pkl
Output rows: 291643
Output columns: 38
Applicants represented: 291643
Remaining numerical missing values: 519170


## Main feature engineering results

This notebook consolidated split payments and built 37 applicant-level features from the cleaned instalment-payment data, covering late payments, underpayments, and recent 12-month behaviour.

All checks passed, and no features needed to be removed. The feature table has 38 columns for the 291,643 applicants with instalment history. The next step is to build features from the credit-card data.
